# Lab 12: Build a Simple LLM Agent Using Phi Data Framework


In [3]:
!pip install agno groq duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.8 MB/s eta 0:00:00


## Task 1: Basic Agent with Web Search Tool


In [7]:
from agno.agent import Agent
from agno.models.groq import Groq

groq_api_key = "api"

web_agent = Agent(
    name="Web Research Agent",
    model=Groq(id="llama-3.1-8b-instant", api_key=groq_api_key),
    instructions="Answer clearly and concisely."
)

web_agent.print_response(
    "Explain recent trends in large language models in simple points.",
    stream=True
)


Output()

## Task 2: Agent with Custom Python Tools


In [20]:
from agno.agent import Agent
from agno.models.groq import Groq
from agno.tools import tool

@tool(show_result=True, stop_after_tool_call=True)
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate BMI given weight in kilograms and height in meters."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.1f} ({category})"

@tool(show_result=True, stop_after_tool_call=True)
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between km and miles, Celsius and Fahrenheit, or kg and pounds."""
    conversions = {
        ("km", "miles"): lambda x: x * 0.621371,
        ("miles", "km"): lambda x: x * 1.60934,
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
        ("kg", "pounds"): lambda x: x * 2.20462,
        ("pounds", "kg"): lambda x: x * 0.453592,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    return f"Conversion from {from_unit} to {to_unit} not supported."

groq_api_key = "api"

utility_agent = Agent(
    name="Utility Agent",
    model=Groq(id="llama-3.1-8b-instant", api_key=groq_api_key),
    tools=[calculate_bmi, unit_converter],
    instructions="Use exactly one appropriate tool to answer the user's request.",
    tool_call_limit=1
)

utility_agent.print_response("Calculate BMI for 70 kg and 1.75 m.", stream=True)
utility_agent.print_response("Convert 100 km to miles.", stream=True)


Output()

Output()

## Task 3: Agent with Memory


In [18]:
from agno.agent import Agent
from agno.models.groq import Groq
from agno.db.sqlite import SqliteDb

groq_api_key = "yourapi_key"

db = SqliteDb(db_file="agent_memory.db")

memory_agent = Agent(
    name="Personal Assistant",
    model=Groq(id="llama-3.1-8b-instant", api_key=groq_api_key),
    db=db,
    add_history_to_context=True,
    num_history_runs=3,
    instructions="You are a personal assistant. Remember what the user tells you during this chat.",
    markdown=True
)

session_id = "user_001"

memory_agent.print_response(
    "My name is Alex and I'm studying AI at RVU.",
    session_id=session_id,
    stream=True
)

memory_agent.print_response(
    "What is my name and where am I studying?",
    session_id=session_id,
    stream=True
)


Output()

Output()

## Task 4: Multi-Agent Team


In [25]:
from agno.agent import Agent
from agno.models.groq import Groq

groq_api_key = "api"

researcher = Agent(
    name="Researcher",
    model=Groq(id="llama-3.1-8b-instant", api_key=groq_api_key),
    instructions="Give exactly 5 factual bullet points about the topic."
)

writer = Agent(
    name="Writer",
    model=Groq(id="llama-3.1-8b-instant", api_key=groq_api_key),
    instructions="Write a short article from the bullet points provided."
)

research_notes = researcher.run("How does RAG improve LLM accuracy?")
print("Research Notes:\n")
print(research_notes.content)

article = writer.run(
    f"Write a short article using these research notes:\n\n{research_notes.content}"
)
print("\nArticle:\n")
print(article.content)


Research Notes:

RAG stands for Retrieval-Augmented Generation. It's a technique used to enhance the accuracy of Large Language Models (LLMs) by retrieving relevant information from a knowledge base, known as a document store or database, and incorporating it into the model's output. Here are 5 factual bullet points about how RAG improves LLM accuracy:

• **Multimodal interaction**: RAG enables the interaction between the LLM and the knowledge base at multiple levels (token, sentence, etc.), allowing the model to access and incorporate more complex knowledge and information from the retrieval system.

• **Knowledge consolidation**: By using a knowledge graph or document store, RAG allows the model to accumulate and consolidate knowledge across multiple instances, reducing its dependence on memorization-based approaches to learning and improving its robustness.

• **Efficient learning**: RAG enables the model to selectively retrieve relevant information from the knowledge base, enabling